# Amazon ML Challenge 2026: Business Entity Resolution


### 1. Environment & Imports


In [ ]:
# !pip install -q rapidfuzz jellyfish lightgbm
import os, sys, json, math, re, pickle, unicodedata, zipfile, warnings
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import rapidfuzz
from rapidfuzz import fuzz, distance
import jellyfish
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import lightgbm as lgb

warnings.filterwarnings('ignore')
print('Environment ready.')


### 2. Configuration & Paths


In [ ]:
"""
Central configuration for Entity Resolution system.
"""
from pathlib import Path

# Base Paths
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path(".").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"
OUTPUT_DIR = PROJECT_ROOT / "output"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILS_DIR = PROJECT_ROOT / "utils"

# Standard Filenames
TRAIN_SOURCE1 = "train_source1.tsv"
TRAIN_SOURCE2 = "train_source2.tsv"
TRAIN_SOURCE3 = "train_source3.tsv"
TRAIN_GROUND_TRUTH = "train_ground_truth.tsv"

TEST_SOURCE1 = "test_source1.tsv"
TEST_SOURCE2 = "test_source2.tsv"
TEST_SOURCE3 = "test_source3.tsv"

MATCHING_RESULTS_FILE = "matching_results.tsv"
CANDIDATE_PAIRS_FILE = "candidate_pairs.tsv"
INTERNAL_CANDIDATE_FILE = "candidate_pairs_internal.tsv"

MODEL_FILE = "matcher.pkl"
THRESHOLD_FILE = "threshold.json"

# Schemas
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
GROUND_TRUTH_COLUMNS = ["source1_entity_id", "matched_entity_ids"]
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]
INTERNAL_CANDIDATE_HEADER = [
    "source1_entity_id",
    "candidate_entity_id",
    "candidate_source",
    "blocking_reasons",
]

# Pipeline Settings
RANDOM_STATE = 42
F_BETA = 0.5
DEFAULT_MATCH_THRESHOLD = 0.50
MAX_CANDIDATES_PER_S1 = 100
HARD_NEGATIVES_PER_POS = 8


### 3. Data Validation & Contracts


In [ ]:
"""
Schema and data validation for Entity Resolution pipeline.
"""
from typing import Dict, List, Optional, Set, Tuple
import pandas as pd



class ValidationError(Exception):
    """Custom exception raised when data contract is violated."""
    pass


def validate_source_dataframe(
    df: pd.DataFrame,
    expected_prefix: str,
    dataset_name: str = "source",
    strict: bool = True,
) -> Dict[str, any]:
    """
    Validates a source DataFrame against the data contract:
    1. Required columns exist.
    2. entity_id uniqueness.
    3. Correct ID prefix (S1-, S2-, S3-).
    4. Reports null rates and duplicate rows.
    """
    issues = []
    
    # Check columns
    missing_cols = [c for c in SOURCE_COLUMNS if c not in df.columns]
    if missing_cols:
        issues.append(f"Missing required columns in {dataset_name}: {missing_cols}")

    if "entity_id" in df.columns:
        # Check uniqueness
        dup_ids = df[df["entity_id"].duplicated()]["entity_id"].tolist()
        if dup_ids:
            issues.append(
                f"Duplicate entity_id found in {dataset_name} ({len(dup_ids)} duplicates, e.g. {dup_ids[:3]})"
            )

        # Check prefix
        invalid_prefix_mask = ~df["entity_id"].astype(str).str.startswith(expected_prefix)
        invalid_prefix_count = invalid_prefix_mask.sum()
        if invalid_prefix_count > 0:
            sample_invalid = df.loc[invalid_prefix_mask, "entity_id"].head(3).tolist()
            issues.append(
                f"Found {invalid_prefix_count} entity_ids not starting with '{expected_prefix}' in {dataset_name}: {sample_invalid}"
            )

    # Check null rates
    null_counts = df[SOURCE_COLUMNS].isna().sum().to_dict()
    total_rows = len(df)
    
    diagnostics = {
        "dataset_name": dataset_name,
        "total_rows": total_rows,
        "null_counts": null_counts,
        "unique_entities": df["entity_id"].nunique() if "entity_id" in df.columns else 0,
        "countries": df["country"].unique().tolist() if "country" in df.columns else [],
        "issues": issues,
    }

    if issues and strict:
        raise ValidationError("; ".join(issues))

    return diagnostics


def parse_ground_truth(
    gt_df: pd.DataFrame,
    valid_s1_ids: Optional[Set[str]] = None,
    valid_s2_s3_ids: Optional[Set[str]] = None,
    strict: bool = True,
) -> Tuple[Dict[str, List[str]], Dict[str, any]]:
    """
    Parses and validates ground truth:
    - Returns mapping: source1_entity_id -> list of matched S2-/S3- entity_ids
    - Empty string -> empty list (singleton)
    - Validates no duplicate S1 rows, no duplicate IDs in match list, no S1 self-matches.
    - Gracefully handles unknown S2/S3 IDs without crashing: records them as warnings
      and filters them so downstream processing remains stable.
    """
    issues = []
    warnings = []
    
    missing_cols = [c for c in GROUND_TRUTH_COLUMNS if c not in gt_df.columns]
    if missing_cols:
        raise ValidationError(f"Ground truth missing required columns: {missing_cols}")

    s1_col = gt_df["source1_entity_id"].astype(str)
    dup_s1 = gt_df[s1_col.duplicated()]["source1_entity_id"].tolist()
    if dup_s1:
        issues.append(f"Duplicate source1_entity_id rows in ground truth: {dup_s1[:5]}")

    # Initialize with all valid S1 entities if provided
    gt_map: Dict[str, List[str]] = {s1: [] for s1 in valid_s1_ids} if valid_s1_ids else {}
    
    total_raw_matches = 0
    total_valid_matches = 0
    singletons = 0
    self_matches = []
    unknown_ids = []

    for _, row in gt_df.iterrows():
        s1_id = str(row["source1_entity_id"]).strip()
        
        # If valid_s1_ids is restricted and s1_id is not in it, skip
        if valid_s1_ids is not None and s1_id not in valid_s1_ids:
            continue

        raw_matches = str(row["matched_entity_ids"]).strip() if pd.notna(row["matched_entity_ids"]) else ""

        if not raw_matches or raw_matches.lower() in ("none", "nan"):
            matched_list = []
            singletons += 1
            gt_map[s1_id] = []
        else:
            matched_list = [m.strip() for m in raw_matches.split(",") if m.strip()]
            
            # Check for intra-row duplicates
            if len(matched_list) != len(set(matched_list)):
                warnings.append(f"Intra-row duplicate IDs for S1 ID '{s1_id}': {matched_list}")
                matched_list = list(dict.fromkeys(matched_list))
            
            valid_matches_for_s1 = []
            for mid in matched_list:
                total_raw_matches += 1
                if mid.startswith("S1-"):
                    self_matches.append(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    issues.append(f"Invalid match ID prefix: '{mid}' for '{s1_id}'")
                elif valid_s2_s3_ids is not None and mid not in valid_s2_s3_ids:
                    unknown_ids.append(mid)
                else:
                    valid_matches_for_s1.append(mid)

            total_valid_matches += len(valid_matches_for_s1)
            gt_map[s1_id] = valid_matches_for_s1

    if self_matches:
        issues.append(f"Ground truth contains Source 1 self-matches: {self_matches[:5]}")

    if unknown_ids:
        warnings.append(
            f"{len(unknown_ids)} ground truth ID(s) not found in S2/S3 pool (e.g. {unknown_ids[:5]}). "
            "These have been filtered out of active matching."
        )

    stats = {
        "total_s1_entities": len(gt_map),
        "total_raw_matches": total_raw_matches,
        "total_valid_matches": total_valid_matches,
        "unknown_ids_count": len(unknown_ids),
        "singletons_count": sum(1 for v in gt_map.values() if len(v) == 0),
        "multi_match_count": sum(1 for v in gt_map.values() if len(v) > 1),
        "issues": issues,
        "warnings": warnings,
    }

    if warnings:
        for w in warnings:
            print(f"VALIDATION WARNING: {w}")

    if issues and strict:
        raise ValidationError("; ".join(issues))

    return gt_map, stats


### 4. Data I/O & Loaders


In [ ]:
"""
Input/Output utilities for TSV files and canonical dataset representation.
"""
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import pandas as pd




def read_tsv(filepath: Union[str, Path]) -> pd.DataFrame:
    """
    Safely reads a TSV file with tab separator, preserving literal commas,
    avoiding silent column collapsing, and keeping empty strings.
    """
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")
    
    df = pd.read_csv(
        filepath,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8",
    )
    return df


def write_tsv(df: pd.DataFrame, filepath: Union[str, Path], columns: Optional[List[str]] = None) -> None:
    """
    Safely writes a DataFrame to a TSV file with tab separator and UTF-8 encoding.
    """
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if columns is not None:
        df = df[columns]
    df.to_csv(filepath, sep="\t", index=False, encoding="utf-8")


def load_source_file(
    filepath: Union[str, Path],
    source_prefix: str,
    strict: bool = True,
) -> pd.DataFrame:
    """
    Loads one source file (S1, S2, or S3), validates schema, and attaches canonical
    metadata columns (source: 'S1'/'S2'/'S3', record_role: 'query'/'candidate').
    Preserves raw text fields unchanged.
    """
    df = read_tsv(filepath)
    validate_source_dataframe(df, expected_prefix=source_prefix, dataset_name=Path(filepath).name, strict=strict)
    
    # Add canonical representation columns
    df["source"] = source_prefix.replace("-", "")
    df["record_role"] = "query" if source_prefix.startswith("S1") else "candidate"
    
    # Ensure all required columns are string and non-null
    for col in SOURCE_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()
            
    return df


def load_ground_truth_file(
    filepath: Union[str, Path],
    valid_s1_ids: Optional[set] = None,
    valid_s2_s3_ids: Optional[set] = None,
    strict: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, List[str]], Dict[str, any]]:
    """
    Loads ground truth TSV, validates, and returns (df, gt_map, stats).
    """
    df = read_tsv(filepath)
    gt_map, stats = parse_ground_truth(df, valid_s1_ids=valid_s1_ids, valid_s2_s3_ids=valid_s2_s3_ids, strict=strict)
    return df, gt_map, stats


def load_split_datasets(
    data_dir: Union[str, Path],
    is_train: bool = True,
    strict: bool = True,
) -> Dict[str, any]:
    """
    Loads all source files (and ground truth if is_train=True) for a split.
    """
    data_dir = Path(data_dir)
    if is_train:
        s1 = load_source_file(data_dir / TRAIN_SOURCE1, source_prefix="S1-", strict=strict)
        s2 = load_source_file(data_dir / TRAIN_SOURCE2, source_prefix="S2-", strict=strict)
        s3 = load_source_file(data_dir / TRAIN_SOURCE3, source_prefix="S3-", strict=strict)
        
        valid_s1 = set(s1["entity_id"])
        valid_s2_s3 = set(s2["entity_id"]) | set(s3["entity_id"])
        gt_df, gt_map, gt_stats = load_ground_truth_file(
            data_dir / TRAIN_GROUND_TRUTH,
            valid_s1_ids=valid_s1,
            valid_s2_s3_ids=valid_s2_s3,
            strict=strict,
        )
        return {
            "source1": s1,
            "source2": s2,
            "source3": s3,
            "ground_truth_df": gt_df,
            "ground_truth_map": gt_map,
            "ground_truth_stats": gt_stats,
        }
    else:
        s1 = load_source_file(data_dir / TEST_SOURCE1, source_prefix="S1-", strict=strict)
        s2 = load_source_file(data_dir / TEST_SOURCE2, source_prefix="S2-", strict=strict)
        s3 = load_source_file(data_dir / TEST_SOURCE3, source_prefix="S3-", strict=strict)
        return {
            "source1": s1,
            "source2": s2,
            "source3": s3,
        }


### 5. Text Normalization


In [ ]:
"""
Text and entity name normalization module.
"""
import re
import unicodedata
from typing import Dict, List, Set, Tuple
import jellyfish

# Generic stop words and legal suffix variants
LEGAL_SUFFIX_MAP = {
    r"\bpvt\.?\s*ltd\.?\b": "private limited",
    r"\bpvt\.?\b": "private",
    r"\bltd\.?\b": "limited",
    r"\bcorp\.?\b": "corporation",
    r"\binc\.?\b": "incorporated",
    r"\bllc\.?\b": "llc",
    r"\bl\.l\.c\.?\b": "llc",
    r"\bco\.?\b": "company",
    r"\bco\s+ltd\.?\b": "company limited",
}

LEGAL_SUFFIX_TOKENS = {
    "private", "limited", "pvt", "ltd", "corp", "corporation",
    "inc", "incorporated", "llc", "company", "co", "enterprises",
    "enterprise", "group", "holdings", "holding", "services",
    "solutions", "international", "global", "industries", "industry"
}

GENERIC_NAME_STOPWORDS = {
    "the", "and", "of", "in", "for", "at", "by", "a", "an", "&"
}


def unicode_clean(text: str) -> str:
    """Normalize unicode characters and strip accents."""
    if not text:
        return ""
    text = str(text)
    norm = unicodedata.normalize("NFKD", text)
    return "".join(c for c in norm if not unicodedata.combining(c))


def normalize_country(country: str) -> str:
    """
    Normalizes country string format (whitespace, casing) without hardcoding
    or limiting to any fixed vocabulary. Unseen countries like 'France' pass through.
    """
    if not country:
        return ""
    clean = unicode_clean(country).strip()
    return re.sub(r"\s+", " ", clean).upper()


def normalize_business_name(raw_name: str) -> Dict[str, any]:
    """
    Normalizes business name preserving raw text.
    Returns dictionary with normalized text, core tokens, legal suffixes,
    and phonetic encodings.
    """
    if not raw_name:
        return {
            "name_raw": "",
            "name_norm": "",
            "name_tokens": [],
            "core_tokens": [],
            "legal_suffix": "",
            "phonetic_key": "",
        }

    # 1. Unicode normalize & lowercase
    text = unicode_clean(raw_name).lower()

    # 2. Replace & with 'and'
    text = re.sub(r"&", " and ", text)

    # 3. Standardize legal suffixes
    legal_found = []
    for pattern, replacement in LEGAL_SUFFIX_MAP.items():
        if re.search(pattern, text):
            legal_found.append(replacement)
            text = re.sub(pattern, f" {replacement} ", text)

    # 4. Strip punctuation, keeping alphanumeric
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 5. Tokenize
    all_tokens = [t for t in text.split() if t]
    
    # 6. Core tokens (strip stopwords and generic legal words for blocking/similarity)
    core_tokens = [
        t for t in all_tokens 
        if t not in GENERIC_NAME_STOPWORDS and t not in LEGAL_SUFFIX_TOKENS
    ]
    if not core_tokens:
        # Fall back to all tokens if all were filtered
        core_tokens = [t for t in all_tokens if t not in GENERIC_NAME_STOPWORDS] or all_tokens

    # 7. Phonetic keys using Metaphone
    phonetics = []
    for t in core_tokens:
        if t.isalpha():
            code = jellyfish.metaphone(t)
            if code:
                phonetics.append(code)
    phonetic_key = " ".join(phonetics)

    return {
        "name_raw": raw_name,
        "name_norm": text,
        "name_tokens": all_tokens,
        "core_tokens": core_tokens,
        "legal_suffix": " ".join(legal_found),
        "phonetic_key": phonetic_key,
    }


### 6. Address Parser


In [ ]:
"""
Address normalization and structured sub-field extraction module.
"""
import re
from typing import Dict, List, Optional

# Common address abbreviation expansions
ADDRESS_ABBREV_MAP = {
    r"\brd\.?\b": "road",
    r"\bst\.?\b": "street",
    r"\bave\.?\b": "avenue",
    r"\bblvd\.?\b": "boulevard",
    r"\bdr\.?\b": "drive",
    r"\bln\.?\b": "lane",
    r"\bhwy\.?\b": "highway",
    r"\bste\.?\b": "suite",
    r"\bapt\.?\b": "apartment",
    r"\bbldg\.?\b": "building",
    r"\bflr?\.?\b": "floor",
    r"\bindl\.?\b": "industrial",
    r"\bind\.?\b": "industrial",
    r"\bopp\.?\b": "opposite",
    r"\bnr\.?\b": "near",
    r"\bpk\.?\b": "park",
    r"\bct\.?\b": "court",
    r"\bsq\.?\b": "square",
    r"\bctr\.?\b": "center",
    r"\bdept\.?\b": "department",
    r"\bmt\.?\b": "mount",
    r"\bn\.\b": "north",
    r"\bs\.\b": "south",
    r"\be\.\b": "east",
    r"\bw\.\b": "west",
}

GENERIC_ADDR_TOKENS = {
    "road", "street", "avenue", "drive", "lane", "suite", "floor",
    "building", "block", "plot", "near", "opposite", "area", "phase",
    "sector", "nagar", "marg", "cross", "main", "east", "west",
    "north", "south", "in", "at", "to", "and", "of", "the", "rue", "de", "la"
}


def extract_postal_code(raw_addr: str) -> str:
    """
    Extracts postal code (6-digit Indian PIN, 5-digit US Zip or French code).
    Avoids matching generic short numbers.
    """
    # 6-digit Indian PIN (often starts with 1-8, but any 6-digit isolated number)
    match_6 = re.findall(r"\b[1-9]\d{5}\b", raw_addr)
    if match_6:
        return match_6[-1]

    # 5-digit US Zip code or French postal code
    match_5 = re.findall(r"\b\d{5}\b", raw_addr)
    if match_5:
        return match_5[-1]

    return ""


def extract_house_or_building(raw_addr: str) -> str:
    """
    Extracts building / house / plot / block numbers.
    Prioritizes leading street/house numbers, followed by plot/block designations.
    """
    # Leading number pattern: e.g., '1200 Market', '12/4 Gandhi Road', '15 Rue'
    lead_match = re.search(r"^\s*([0-9]+[a-z]?(?:/[0-9]+)?)\b", raw_addr, re.IGNORECASE)
    if lead_match:
        return lead_match.group(1).lower()

    # Plot or Block pattern: e.g., 'plot 45', 'block a', 'shop 12'
    plot_match = re.search(r"\b(plot|block|shop|flat|suite|ste|no\.?)\s*([a-z0-9/-]+)\b", raw_addr, re.IGNORECASE)
    if plot_match:
        return f"{plot_match.group(1).lower()} {plot_match.group(2).lower()}"

    # Comma-delimited number token: e.g., ', 797, Lake Town'
    comma_num = re.search(r",\s*([0-9]+[a-z]?)\s*,", raw_addr)
    if comma_num:
        return comma_num.group(1).lower()

    return ""


def extract_landmark(raw_addr: str) -> str:
    """
    Extracts landmark descriptions common in Indian addresses (e.g. 'Near SBI ATM').
    """
    match = re.search(r"\b(near|opp|opposite|behind|beside)\s+([a-zA-Z0-9\s]{3,35}?)(?:,|$|\b(?:road|street|nagar|marg)\b)", raw_addr, re.IGNORECASE)
    if match:
        return re.sub(r"\s+", " ", match.group(0).lower()).strip()
    return ""


def normalize_address(raw_addr: str) -> Dict[str, any]:
    """
    Normalizes address text, standardizes street abbreviations, extracts
    postal code, building number, landmark, and distinctive tokens.
    """
    if not raw_addr:
        return {
            "addr_raw": "",
            "addr_norm": "",
            "addr_tokens": [],
            "postal_code": "",
            "house_number": "",
            "landmark": "",
            "distinctive_tokens": [],
        }

    # Extract structured fields before heavy stripping
    postal_code = extract_postal_code(raw_addr)
    house_number = extract_house_or_building(raw_addr)
    landmark = extract_landmark(raw_addr)

    # Unicode & lowercase
    text = unicode_clean(raw_addr).lower()

    # Expand abbreviations
    for pattern, replacement in ADDRESS_ABBREV_MAP.items():
        text = re.sub(pattern, f" {replacement} ", text)

    # Strip punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    all_tokens = [t for t in text.split() if t]
    distinctive_tokens = [
        t for t in all_tokens
        if t not in GENERIC_ADDR_TOKENS and not (t.isdigit() and len(t) < 3)
    ]

    return {
        "addr_raw": raw_addr,
        "addr_norm": text,
        "addr_tokens": all_tokens,
        "postal_code": postal_code,
        "house_number": house_number,
        "landmark": landmark,
        "distinctive_tokens": distinctive_tokens,
    }


### 7. Inverted Index Blocking Engine


In [ ]:
"""
Multi-rule inverted-index blocking engine for candidate pair generation.
"""
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple
import pandas as pd




def get_char_ngrams(text: str, n: int = 3) -> Set[str]:
    """Generates character n-grams from normalized text."""
    clean = "".join(text.split())
    if len(clean) < n:
        return {clean} if clean else set()
    return {clean[i : i + n] for i in range(len(clean) - n + 1)}


class BlockingEngine:
    """
    High-recall blocking engine using multiple inverted indexes.
    Indexed over S2 and S3 candidate entities.
    """

    def __init__(
        self,
        max_candidates_per_query: int = MAX_CANDIDATES_PER_S1,
        max_token_postings: int = 1000,
    ):
        self.max_candidates_per_query = max_candidates_per_query
        self.max_token_postings = max_token_postings

        # Inverted indexes: token -> list of candidate entity_ids
        self.exact_name_index = defaultdict(list)
        self.name_token_index = defaultdict(list)
        self.phonetic_index = defaultdict(list)
        self.postal_index = defaultdict(list)
        self.addr_token_index = defaultdict(list)
        self.ngram_index = defaultdict(list)

        # Storage for candidate metadata
        self.candidates_meta: Dict[str, Dict[str, any]] = {}

    def fit(self, s2_df: pd.DataFrame, s3_df: pd.DataFrame) -> "BlockingEngine":
        """Builds inverted indexes across all S2 and S3 records."""
        combined_pool = pd.concat([s2_df, s3_df], ignore_index=True)

        for _, row in combined_pool.iterrows():
            cid = str(row["entity_id"]).strip()
            raw_name = str(row["business_name"])
            raw_addr = str(row["business_address"])
            raw_country = str(row["country"])

            norm_name = normalize_business_name(raw_name)
            norm_addr = normalize_address(raw_addr)
            country = normalize_country(raw_country)

            self.candidates_meta[cid] = {
                "entity_id": cid,
                "source": "S2" if cid.startswith("S2-") else "S3",
                "name_norm": norm_name["name_norm"],
                "core_tokens": norm_name["core_tokens"],
                "phonetic_key": norm_name["phonetic_key"],
                "postal_code": norm_addr["postal_code"],
                "addr_tokens": norm_addr["distinctive_tokens"],
                "country": country,
            }

            # 1. Exact normalized name
            if norm_name["name_norm"]:
                self.exact_name_index[norm_name["name_norm"]].append(cid)

            # 2. Informative name tokens
            for token in norm_name["core_tokens"]:
                if len(token) >= 3:
                    self.name_token_index[token].append(cid)

            # 3. Phonetic tokens
            if norm_name["phonetic_key"]:
                for p_code in norm_name["phonetic_key"].split():
                    if len(p_code) >= 2:
                        self.phonetic_index[p_code].append(cid)

            # 4. Postal code (when present)
            if norm_addr["postal_code"]:
                self.postal_index[norm_addr["postal_code"]].append(cid)

            # 5. Distinctive address tokens
            for atoken in norm_addr["distinctive_tokens"]:
                if len(atoken) >= 4:
                    self.addr_token_index[atoken].append(cid)

            # 6. Character 3-grams of name
            for gram in get_char_ngrams(norm_name["name_norm"], n=3):
                self.ngram_index[gram].append(cid)

        return self

    def query_entity(self, s1_row: pd.Series) -> Dict[str, Set[str]]:
        """
        Retrieves candidate entity_ids and tracks blocking reasons.
        Returns: {candidate_id: {reason1, reason2, ...}}
        """
        raw_name = str(s1_row["business_name"])
        raw_addr = str(s1_row["business_address"])
        raw_country = str(s1_row.get("country", ""))

        norm_name = normalize_business_name(raw_name)
        norm_addr = normalize_address(raw_addr)
        country = normalize_country(raw_country)

        candidates_reasons: Dict[str, Set[str]] = defaultdict(set)

        # Rule 1: Exact Name
        if norm_name["name_norm"] and norm_name["name_norm"] in self.exact_name_index:
            for cid in self.exact_name_index[norm_name["name_norm"]]:
                candidates_reasons[cid].add("exact_name")

        # Rule 2: Informative Name Token Overlap
        token_hits = defaultdict(int)
        for token in norm_name["core_tokens"]:
            if len(token) >= 3 and token in self.name_token_index:
                postings = self.name_token_index[token]
                if len(postings) <= self.max_token_postings:
                    for cid in postings:
                        token_hits[cid] += 1

        min_token_overlap = 1 if len(norm_name["core_tokens"]) <= 2 else 2
        for cid, hits in token_hits.items():
            if hits >= min_token_overlap:
                candidates_reasons[cid].add("name_token_overlap")

        # Rule 3: Phonetic Matching
        if norm_name["phonetic_key"]:
            phon_hits = defaultdict(int)
            for p_code in norm_name["phonetic_key"].split():
                if len(p_code) >= 2 and p_code in self.phonetic_index:
                    postings = self.phonetic_index[p_code]
                    if len(postings) <= self.max_token_postings:
                        for cid in postings:
                            phon_hits[cid] += 1
            for cid, hits in phon_hits.items():
                if hits >= 1:
                    candidates_reasons[cid].add("phonetic_name")

        # Rule 4: Postal Code match
        if norm_addr["postal_code"] and norm_addr["postal_code"] in self.postal_index:
            for cid in self.postal_index[norm_addr["postal_code"]]:
                # Require some token or country agreement to avoid pure zip code flood
                cand_meta = self.candidates_meta.get(cid, {})
                if cand_meta.get("country") == country or not country:
                    candidates_reasons[cid].add("postal_code")

        # Rule 5: Address Token Overlap
        addr_hits = defaultdict(int)
        for atoken in norm_addr["distinctive_tokens"]:
            if len(atoken) >= 4 and atoken in self.addr_token_index:
                postings = self.addr_token_index[atoken]
                if len(postings) <= self.max_token_postings:
                    for cid in postings:
                        addr_hits[cid] += 1
        for cid, hits in addr_hits.items():
            if hits >= 2:
                candidates_reasons[cid].add("address_token_overlap")

        # Rule 6: Name Character 3-Gram Overlap
        s1_grams = get_char_ngrams(norm_name["name_norm"], n=3)
        if len(s1_grams) >= 3:
            gram_hits = defaultdict(int)
            for g in s1_grams:
                if g in self.ngram_index:
                    postings = self.ngram_index[g]
                    if len(postings) <= self.max_token_postings:
                        for cid in postings:
                            gram_hits[cid] += 1
            threshold_hits = max(3, int(len(s1_grams) * 0.4))
            for cid, hits in gram_hits.items():
                if hits >= threshold_hits:
                    candidates_reasons[cid].add("char_ngram")

        # Prioritize candidates and cap per query
        if len(candidates_reasons) > self.max_candidates_per_query:
            # Sort by number of independent blocking reasons descending
            sorted_candidates = sorted(
                candidates_reasons.items(),
                key=lambda item: len(item[1]),
                reverse=True,
            )
            candidates_reasons = dict(sorted_candidates[: self.max_candidates_per_query])

        return candidates_reasons

    def generate_candidates(
        self, s1_df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Generates candidates for every S1 entity in s1_df.
        Returns:
            - internal_df: 4 columns (source1_entity_id, candidate_entity_id, candidate_source, blocking_reasons)
            - competition_df: 2 columns (source1_entity_id, candidate_entity_ids)
        """
        internal_rows = []
        comp_rows = []

        for _, s1_row in s1_df.iterrows():
            s1_id = str(s1_row["entity_id"]).strip()
            cands_dict = self.query_entity(s1_row)

            cand_id_list = []
            for cid, reasons in cands_dict.items():
                cand_id_list.append(cid)
                source_tag = "S2" if cid.startswith("S2-") else "S3"
                internal_rows.append({
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": cid,
                    "candidate_source": source_tag,
                    "blocking_reasons": ",".join(sorted(reasons)),
                })

            comp_rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_ids": ",".join(cand_id_list),
            })

        internal_df = pd.DataFrame(internal_rows, columns=INTERNAL_CANDIDATE_HEADER)
        competition_df = pd.DataFrame(comp_rows, columns=CANDIDATE_HEADER)

        return internal_df, competition_df


### 8. Labeled Pair Construction & Grouped Splitting


In [ ]:
"""
Training pair construction: positives, hard negatives, and grouped train/val split.
"""
from typing import Dict, List, Set, Tuple
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit



def build_labeled_training_pairs(
    internal_candidates_df: pd.DataFrame,
    gt_map: Dict[str, List[str]],
    hard_negatives_per_pos: int = HARD_NEGATIVES_PER_POS,
    random_state: int = RANDOM_STATE,
) -> Tuple[pd.DataFrame, Dict[str, any]]:
    """
    Constructs labeled pairs from internal candidate table and ground truth:
    - Positive: (s1_id, cand_id) where cand_id in gt_map[s1_id]
    - Hard Negative: (s1_id, cand_id) retrieved by blocking but not in gt_map[s1_id]
    - Computes blocking recall ceiling: survived_positives / total_gt_positives.
    """
    total_gt_positives = sum(len(matches) for matches in gt_map.values())
    gt_pos_set = {(s1, mid) for s1, matches in gt_map.items() for mid in matches}

    positive_rows = []
    negative_rows = []

    # Map candidate rows by s1_id for hard negative sampling
    s1_to_candidates = internal_candidates_df.groupby("source1_entity_id")

    rng = np.random.default_rng(random_state)
    survived_positives = 0

    for s1_id, group in s1_to_candidates:
        gt_matches = set(gt_map.get(s1_id, []))
        
        pos_in_group = []
        neg_in_group = []

        for _, row in group.iterrows():
            cand_id = row["candidate_entity_id"]
            pair = (s1_id, cand_id)
            pair_dict = row.to_dict()

            if cand_id in gt_matches:
                pair_dict["label"] = 1
                pos_in_group.append(pair_dict)
                survived_positives += 1
            else:
                pair_dict["label"] = 0
                neg_in_group.append(pair_dict)

        positive_rows.extend(pos_in_group)

        # Hard negative subsampling if negatives outnumber target ratio
        num_pos = len(pos_in_group)
        target_negs = max(hard_negatives_per_pos, num_pos * hard_negatives_per_pos)
        
        if len(neg_in_group) > target_negs:
            # Sort by number of blocking reasons (hardest negatives first)
            neg_in_group.sort(
                key=lambda x: len(str(x.get("blocking_reasons", "")).split(",")),
                reverse=True,
            )
            selected_negs = neg_in_group[:target_negs]
        else:
            selected_negs = neg_in_group

        negative_rows.extend(selected_negs)

    all_pairs = positive_rows + negative_rows
    labeled_df = pd.DataFrame(all_pairs)

    recall_ceiling = (
        float(survived_positives / total_gt_positives) if total_gt_positives > 0 else 1.0
    )

    diagnostics = {
        "total_gt_positives": total_gt_positives,
        "survived_positives": survived_positives,
        "missed_positives": total_gt_positives - survived_positives,
        "blocking_recall_ceiling": recall_ceiling,
        "total_labeled_pairs": len(labeled_df),
        "positive_count": len(positive_rows),
        "negative_count": len(negative_rows),
    }

    return labeled_df, diagnostics


def split_grouped_train_val(
    labeled_df: pd.DataFrame,
    val_size: float = 0.2,
    random_state: int = RANDOM_STATE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Splits labeled pairs into Train and Validation sets grouped by source1_entity_id.
    Zero data leakage: all candidate pairs for an S1 entity belong exclusively to either split.
    """
    # Ensure train_df contains both classes if the dataset has multiple classes
    for seed in range(random_state, random_state + 15):
        gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
        groups = labeled_df["source1_entity_id"]
        train_idx, val_idx = next(gss.split(labeled_df, groups=groups))
        train_df = labeled_df.iloc[train_idx].copy().reset_index(drop=True)
        val_df = labeled_df.iloc[val_idx].copy().reset_index(drop=True)
        if train_df["label"].nunique() >= 2 or labeled_df["label"].nunique() < 2:
            break

    # Assert complete disjointness
    train_s1 = set(train_df["source1_entity_id"])
    val_s1 = set(val_df["source1_entity_id"])
    overlap = train_s1.intersection(val_s1)
    if overlap:
        raise ValueError(f"Data leakage detected! S1 IDs in both train and val: {overlap}")

    return train_df, val_df


### 9. Pairwise Feature Engineering


In [ ]:
"""
Pairwise feature engineering for business entity matching.
"""
from collections import Counter
import math
from typing import Dict, List, Set, Union
import numpy as np
import pandas as pd
import rapidfuzz
from rapidfuzz import fuzz, distance



ALL_BLOCKING_RULES = [
    "exact_name",
    "name_token_overlap",
    "phonetic_name",
    "postal_code",
    "address_token_overlap",
    "char_ngram",
]


def token_jaccard(tokens1: List[str], tokens2: List[str]) -> float:
    """Calculates Jaccard similarity between two token lists."""
    s1, s2 = set(tokens1), set(tokens2)
    if not s1 and not s2:
        return 0.0
    intersection = len(s1.intersection(s2))
    union = len(s1.union(s2))
    return float(intersection / union) if union > 0 else 0.0


def char_ngram_cosine(text1: str, text2: str, n: int = 3) -> float:
    """Calculates cosine similarity between character n-gram frequency bags."""
    if not text1 or not text2:
        return 0.0

    def get_counts(s):
        s_clean = "".join(s.split())
        if len(s_clean) < n:
            return Counter([s_clean]) if s_clean else Counter()
        return Counter(s_clean[i : i + n] for i in range(len(s_clean) - n + 1))

    c1, c2 = get_counts(text1), get_counts(text2)
    if not c1 or not c2:
        return 0.0

    dot = sum(c1[g] * c2[g] for g in c1 if g in c2)
    norm1 = math.sqrt(sum(v * v for v in c1.values()))
    norm2 = math.sqrt(sum(v * v for v in c2.values()))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(dot / (norm1 * norm2))


def compute_pair_features(
    s1_record: Dict[str, any],
    s23_record: Dict[str, any],
    blocking_reasons_str: str = "",
) -> Dict[str, float]:
    """
    Computes a comprehensive vector of similarity features for a candidate pair.
    Both records are expected to have:
      business_name, business_address, country (or precomputed normalized representations).
    """
    # Extract or precompute fields
    s1_name_raw = str(s1_record.get("business_name", ""))
    s2_name_raw = str(s23_record.get("business_name", ""))
    s1_addr_raw = str(s1_record.get("business_address", ""))
    s2_addr_raw = str(s23_record.get("business_address", ""))
    s1_country = normalize_country(str(s1_record.get("country", "")))
    s2_country = normalize_country(str(s23_record.get("country", "")))

    s1_name_norm = normalize_business_name(s1_name_raw)
    s2_name_norm = normalize_business_name(s2_name_raw)
    s1_addr_norm = normalize_address(s1_addr_raw)
    s2_addr_norm = normalize_address(s2_addr_raw)

    n1 = s1_name_norm["name_norm"]
    n2 = s2_name_norm["name_norm"]
    a1 = s1_addr_norm["addr_norm"]
    a2 = s2_addr_norm["addr_norm"]

    # 1. Name Similarities
    feat_name_ratio = fuzz.ratio(n1, n2) / 100.0
    feat_name_jw = distance.JaroWinkler.similarity(n1, n2)
    feat_name_token_set = fuzz.token_set_ratio(n1, n2) / 100.0
    feat_name_token_sort = fuzz.token_sort_ratio(n1, n2) / 100.0
    feat_name_ngram_cos = char_ngram_cosine(n1, n2, n=3)

    # Phonetic similarity
    p1 = s1_name_norm["phonetic_key"]
    p2 = s2_name_norm["phonetic_key"]
    feat_phonetic_sim = (fuzz.ratio(p1, p2) / 100.0) if (p1 and p2) else 0.0

    # 2. Address Similarities
    feat_addr_ratio = fuzz.ratio(a1, a2) / 100.0
    feat_addr_jw = distance.JaroWinkler.similarity(a1, a2)
    feat_addr_token_set = fuzz.token_set_ratio(a1, a2) / 100.0
    feat_addr_jaccard = token_jaccard(
        s1_addr_norm["distinctive_tokens"], s2_addr_norm["distinctive_tokens"]
    )
    feat_addr_ngram_cos = char_ngram_cosine(a1, a2, n=3)

    # 3. Structured Subfields (Postal Code, House Number, Country)
    pin1 = s1_addr_norm["postal_code"]
    pin2 = s2_addr_norm["postal_code"]
    if pin1 and pin2:
        feat_same_postal = 1.0 if pin1 == pin2 else 0.0
    else:
        feat_same_postal = -1.0  # missing indicator

    h1 = s1_addr_norm["house_number"]
    h2 = s2_addr_norm["house_number"]
    if h1 and h2:
        feat_same_house = 1.0 if h1 == h2 else 0.0
    else:
        feat_same_house = -1.0  # missing indicator

    # Open-set country equality
    if s1_country and s2_country:
        feat_same_country = 1.0 if s1_country == s2_country else 0.0
    else:
        feat_same_country = 0.5  # unknown

    # 4. Length Discrepancies
    l_n1, l_n2 = len(n1), len(n2)
    feat_name_len_diff_abs = float(abs(l_n1 - l_n2))
    feat_name_len_diff_rel = float(abs(l_n1 - l_n2) / max(l_n1, l_n2, 1))

    l_a1, l_a2 = len(a1), len(a2)
    feat_addr_len_diff_abs = float(abs(l_a1 - l_a2))
    feat_addr_len_diff_rel = float(abs(l_a1 - l_a2) / max(l_a1, l_a2, 1))

    # 5. Multi-hot Blocking Reason Flags
    reasons_set = set(blocking_reasons_str.split(",")) if blocking_reasons_str else set()
    blocking_features = {
        f"block_{rule}": (1.0 if rule in reasons_set else 0.0)
        for rule in ALL_BLOCKING_RULES
    }
    feat_num_blocking_reasons = float(len(reasons_set))

    features = {
        "name_ratio": feat_name_ratio,
        "name_jaro_winkler": feat_name_jw,
        "name_token_set": feat_name_token_set,
        "name_token_sort": feat_name_token_sort,
        "name_ngram_cos": feat_name_ngram_cos,
        "phonetic_sim": feat_phonetic_sim,
        "addr_ratio": feat_addr_ratio,
        "addr_jaro_winkler": feat_addr_jw,
        "addr_token_set": feat_addr_token_set,
        "addr_jaccard": feat_addr_jaccard,
        "addr_ngram_cos": feat_addr_ngram_cos,
        "same_postal": feat_same_postal,
        "same_house": feat_same_house,
        "same_country": feat_same_country,
        "name_len_diff_abs": feat_name_len_diff_abs,
        "name_len_diff_rel": feat_name_len_diff_rel,
        "addr_len_diff_abs": feat_addr_len_diff_abs,
        "addr_len_diff_rel": feat_addr_len_diff_rel,
        "num_blocking_reasons": feat_num_blocking_reasons,
        **blocking_features,
    }

    return features


def extract_features_for_pairs(
    candidate_pairs_df: pd.DataFrame,
    s1_dict: Dict[str, Dict[str, any]],
    s23_dict: Dict[str, Dict[str, any]],
) -> pd.DataFrame:
    """
    Computes pairwise feature matrix for a dataframe containing
    [source1_entity_id, candidate_entity_id, blocking_reasons (optional)].
    """
    feature_rows = []
    
    for _, row in candidate_pairs_df.iterrows():
        s1_id = str(row["source1_entity_id"]).strip()
        cand_id = str(row["candidate_entity_id"]).strip()
        reasons = str(row.get("blocking_reasons", ""))

        s1_rec = s1_dict.get(s1_id, {})
        s23_rec = s23_dict.get(cand_id, {})

        feats = compute_pair_features(s1_rec, s23_rec, blocking_reasons_str=reasons)
        feats["source1_entity_id"] = s1_id
        feats["candidate_entity_id"] = cand_id
        feature_rows.append(feats)

    return pd.DataFrame(feature_rows)


### 10. Evaluation Metrics & Threshold Sweeper


In [ ]:
"""
Evaluation metrics: exact Per-Entity Macro F_0.5 score and pair-level metrics.
"""
from typing import Dict, List, Optional, Set, Tuple
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)



def compute_entity_f_beta(
    true_ids: Set[str],
    pred_ids: Set[str],
    beta: float = F_BETA,
) -> float:
    """
    Computes per-entity F-beta score (default beta=0.5, precision-weighted 2x).
    Singletons:
      - True empty + Pred empty -> 1.0
      - True empty + Pred non-empty -> 0.0
    Non-singletons:
      - True non-empty + Pred empty -> 0.0
      - True non-empty + Pred non-empty -> (1 + beta^2)*P*R / (beta^2*P + R)
    """
    is_true_singleton = len(true_ids) == 0
    is_pred_singleton = len(pred_ids) == 0

    if is_true_singleton:
        return 1.0 if is_pred_singleton else 0.0

    if is_pred_singleton:
        return 0.0

    # Non-singleton case
    tp = len(true_ids.intersection(pred_ids))
    if tp == 0:
        return 0.0

    precision = float(tp / len(pred_ids))
    recall = float(tp / len(true_ids))

    beta_sq = beta * beta
    numerator = (1.0 + beta_sq) * precision * recall
    denominator = (beta_sq * precision) + recall

    return float(numerator / denominator) if denominator > 0 else 0.0


def compute_macro_f_beta(
    gt_map: Dict[str, List[str]],
    pred_map: Dict[str, List[str]],
    all_s1_ids: List[str],
    beta: float = F_BETA,
) -> Tuple[float, Dict[str, any]]:
    """
    Computes the official competition metric:
    Per-entity F_0.5 macro-averaged across ALL Source 1 entities in the evaluation set.
    """
    scores = []
    singleton_correct = 0
    singleton_total = 0
    non_singleton_total = 0

    for s1_id in all_s1_ids:
        true_set = set(gt_map.get(s1_id, []))
        pred_set = set(pred_map.get(s1_id, []))

        score = compute_entity_f_beta(true_set, pred_set, beta=beta)
        scores.append(score)

        if len(true_set) == 0:
            singleton_total += 1
            if len(pred_set) == 0:
                singleton_correct += 1
        else:
            non_singleton_total += 1

    macro_score = float(np.mean(scores)) if scores else 0.0
    
    breakdown = {
        "macro_f_beta": macro_score,
        "total_entities": len(all_s1_ids),
        "singleton_total": singleton_total,
        "singleton_correct": singleton_correct,
        "singleton_accuracy": (singleton_correct / singleton_total) if singleton_total else 1.0,
        "non_singleton_total": non_singleton_total,
    }

    return macro_score, breakdown


def compute_pair_level_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """
    Computes standard binary classification diagnostics on candidate pairs.
    """
    y_pred = (y_prob >= threshold).astype(int)
    
    metrics = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": 0.0,
        "pr_auc": 0.0,
    }
    
    if len(np.unique(y_true)) > 1:
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        except Exception:
            metrics["roc_auc"] = 0.0

        try:
            metrics["pr_auc"] = float(average_precision_score(y_true, y_prob))
        except Exception:
            metrics["pr_auc"] = 0.0

    return metrics


def sweep_decision_threshold(
    val_pairs_df: pd.DataFrame,
    val_probs: np.ndarray,
    gt_map: Dict[str, List[str]],
    val_s1_ids: List[str],
    thresholds: Optional[np.ndarray] = None,
) -> Tuple[float, float, List[Dict[str, float]]]:
    """
    Sweeps decision threshold on held-out validation set to find the threshold
    that directly maximizes the competition metric: per-entity macro F_0.5.
    """
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 37)  # step ~0.025

    val_df_scored = val_pairs_df.copy()
    val_df_scored["score"] = val_probs

    best_thresh = 0.50
    best_macro = -1.0
    sweep_results = []

    for t in thresholds:
        pred_map: Dict[str, List[str]] = {s1: [] for s1 in val_s1_ids}
        
        # Keep pairs where score >= threshold
        accepted = val_df_scored[val_df_scored["score"] >= t]
        grouped = accepted.groupby("source1_entity_id")["candidate_entity_id"].apply(list)
        
        for s1_id, cands in grouped.items():
            if s1_id in pred_map:
                pred_map[s1_id] = cands

        macro_score, _ = compute_macro_f_beta(gt_map, pred_map, val_s1_ids, beta=F_BETA)
        sweep_results.append({"threshold": float(t), "macro_f05": macro_score})

        if macro_score > best_macro:
            best_macro = macro_score
            best_thresh = float(t)

    return best_thresh, best_macro, sweep_results


### 11. Output Exporters


In [ ]:
"""
Output generation for internal audit and competition submissions.
"""
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple, Union
import pandas as pd




def export_candidate_pairs(
    internal_candidate_df: pd.DataFrame,
    s1_entity_ids: List[str],
    output_dir: Union[str, Path],
    save_internal_4col: bool = True,
) -> Tuple[Path, Optional[Path]]:
    """
    Exports candidate pairs to:
    1. output/candidate_pairs.tsv (competition 2-col format)
    2. output/candidate_pairs_internal.tsv (internal 4-col audit format)
    Ensures every S1 entity is present, empty strings for no candidates,
    no intra-row duplicate IDs, and preserved ID strings.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Internal 4-column table
    internal_path = None
    if save_internal_4col:
        internal_path = output_dir / "candidate_pairs_internal.tsv"
        write_tsv(internal_candidate_df, internal_path, columns=INTERNAL_CANDIDATE_HEADER)

    # 2. Derive competition 2-column format
    if len(internal_candidate_df) > 0:
        grouped = (
            internal_candidate_df.groupby("source1_entity_id")["candidate_entity_id"]
            .apply(lambda ids: ",".join(dict.fromkeys(ids)))  # preserve order, eliminate dupes
            .to_dict()
        )
    else:
        grouped = {}

    comp_rows = []
    for s1_id in s1_entity_ids:
        cands = grouped.get(s1_id, "")
        comp_rows.append({"source1_entity_id": s1_id, "candidate_entity_ids": cands})

    comp_df = pd.DataFrame(comp_rows, columns=CANDIDATE_HEADER)
    comp_path = output_dir / "candidate_pairs.tsv"
    write_tsv(comp_df, comp_path)

    return comp_path, internal_path


def export_matching_results(
    predictions_map: Dict[str, List[str]],
    s1_entity_ids: List[str],
    output_dir: Union[str, Path],
) -> Path:
    """
    Exports matching results to output/matching_results.tsv.
    Follows competition rules:
    - Every S1 entity in the split appears exactly once.
    - Singletons have empty string (not None, not NaN).
    - Comma-separated list with no spaces or quoting.
    - Preserves ID strings.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for s1_id in s1_entity_ids:
        matches = predictions_map.get(s1_id, [])
        # Deduplicate while preserving order
        unique_matches = list(dict.fromkeys(matches))
        # Ensure only S2-/S3- IDs
        filtered_matches = [m for m in unique_matches if m.startswith(("S2-", "S3-"))]
        match_str = ",".join(filtered_matches)
        rows.append({"source1_entity_id": s1_id, "matched_entity_ids": match_str})

    df = pd.DataFrame(rows, columns=MATCHING_HEADER)
    matching_path = output_dir / "matching_results.tsv"
    write_tsv(df, matching_path)
    return matching_path


### 12. Training Pipeline Function


In [ ]:
"""
Model training and threshold optimization pipeline.
"""
import argparse
import json
import pickle
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier





FEATURE_COLS = [
    "name_ratio",
    "name_jaro_winkler",
    "name_token_set",
    "name_token_sort",
    "name_ngram_cos",
    "phonetic_sim",
    "addr_ratio",
    "addr_jaro_winkler",
    "addr_token_set",
    "addr_jaccard",
    "addr_ngram_cos",
    "same_postal",
    "same_house",
    "same_country",
    "name_len_diff_abs",
    "name_len_diff_rel",
    "addr_len_diff_abs",
    "addr_len_diff_rel",
    "num_blocking_reasons",
    "block_exact_name",
    "block_name_token_overlap",
    "block_phonetic_name",
    "block_postal_code",
    "block_address_token_overlap",
    "block_char_ngram",
]


def train_entity_matching_system(
    data_dir: Optional[Union[str, Path]] = None,
    data_dict: Optional[Dict[str, any]] = None,
    models_dir: Union[str, Path] = MODELS_DIR,
    val_size: float = 0.25,
    random_state: int = RANDOM_STATE,
) -> Dict[str, any]:
    """
    End-to-end training pipeline:
    1. Load data & ground truth.
    2. Build inverted index blocking on S2+S3 and retrieve S1 candidates.
    3. Measure & log blocking recall ceiling.
    4. Generate labeled pairs with hard negatives.
    5. Grouped train/val split (zero leakage).
    6. Extract pairwise features.
    7. Train baseline (Logistic Regression) & main model (LightGBM).
    8. Sweep validation threshold maximizing per-entity macro F_0.5.
    9. Save model and threshold artifacts.
    """
    models_dir = Path(models_dir)
    models_dir.mkdir(parents=True, exist_ok=True)

    print("=== Step 1: Loading Training Data ===")
    if data_dict is None:
        if data_dir is None:
            raise ValueError("Must provide either data_dir or data_dict")
        data_dict = load_split_datasets(data_dir, is_train=True)

    s1_df = data_dict.get("source1", data_dict.get("train_source1"))
    s2_df = data_dict.get("source2", data_dict.get("train_source2"))
    s3_df = data_dict.get("source3", data_dict.get("train_source3"))
    gt_map = data_dict.get("ground_truth_map", {})

    print(f"Loaded: S1={len(s1_df)}, S2={len(s2_df)}, S3={len(s3_df)}, GT={len(gt_map)}")

    print("=== Step 2: Blocking & Candidate Generation ===")
    blocking_engine = BlockingEngine()
    blocking_engine.fit(s2_df, s3_df)
    internal_cand_df, _ = blocking_engine.generate_candidates(s1_df)

    print("=== Step 3: Labeled Pair Construction & Blocking Recall Ceiling ===")
    labeled_pairs_df, pair_stats = build_labeled_training_pairs(
        internal_cand_df, gt_map, random_state=random_state
    )
    print(f"Blocking Candidate Recall Ceiling: {pair_stats['blocking_recall_ceiling']:.4f} "
          f"({pair_stats['survived_positives']}/{pair_stats['total_gt_positives']} positives survived)")
    print(f"Total labeled pairs: {pair_stats['total_labeled_pairs']} "
          f"(Pos: {pair_stats['positive_count']}, Neg: {pair_stats['negative_count']})")

    if pair_stats["positive_count"] == 0:
        raise ValueError(
            "No positive training pairs found in candidate pool! "
            "None of the ground truth match IDs exist in the loaded S2/S3 files. "
            "If using a slice or sample of the data, ensure the sample is connected."
        )

    print("=== Step 4: Grouped Train/Validation Split ===")
    train_pairs_df, val_pairs_df = split_grouped_train_val(
        labeled_pairs_df, val_size=val_size, random_state=random_state
    )
    val_s1_ids = sorted(list(set(val_pairs_df["source1_entity_id"])))
    print(f"Train pairs: {len(train_pairs_df)}, Val pairs: {len(val_pairs_df)} "
          f"across {len(val_s1_ids)} validation S1 entities")

    print("=== Step 5: Pairwise Feature Engineering ===")
    s1_dict = s1_df.set_index("entity_id").to_dict(orient="index")
    s23_combined = pd.concat([s2_df, s3_df], ignore_index=True)
    s23_dict = s23_combined.set_index("entity_id").to_dict(orient="index")

    X_train_df = extract_features_for_pairs(train_pairs_df, s1_dict, s23_dict)
    X_val_df = extract_features_for_pairs(val_pairs_df, s1_dict, s23_dict)

    X_train = X_train_df[FEATURE_COLS].values
    y_train = train_pairs_df["label"].values
    X_val = X_val_df[FEATURE_COLS].values
    y_val = val_pairs_df["label"].values

    print("=== Step 6: Training Models ===")
    # 6.1 Baseline: Logistic Regression
    lr_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(class_weight="balanced", random_state=random_state, max_iter=1000)),
    ])
    lr_pipeline.fit(X_train, y_train)
    val_lr_probs = lr_pipeline.predict_proba(X_val)[:, 1]
    lr_metrics = compute_pair_level_metrics(y_val, val_lr_probs)
    print(f"Baseline (Logistic Regression) Pair Metrics: {lr_metrics}")

    # 6.2 Main Model: LightGBM
    lgb_model = LGBMClassifier(
        n_estimators=120,
        learning_rate=0.05,
        num_leaves=31,
        random_state=random_state,
        class_weight="balanced",
        verbose=-1,
    )
    lgb_model.fit(X_train, y_train)
    val_lgb_probs = lgb_model.predict_proba(X_val)[:, 1]
    lgb_metrics = compute_pair_level_metrics(y_val, val_lgb_probs)
    print(f"Main Model (LightGBM) Pair Metrics: {lgb_metrics}")

    chosen_model = lgb_model
    chosen_probs = val_lgb_probs
    model_name = "LightGBM"

    print("=== Step 7: Threshold Sweep for Competition Metric (Per-Entity Macro F_0.5) ===")
    best_thresh, best_macro_f05, sweep_results = sweep_decision_threshold(
        val_pairs_df, chosen_probs, gt_map, val_s1_ids
    )
    print(f"Optimal Threshold: {best_thresh:.4f} -> Best Validation Macro F_0.5: {best_macro_f05:.4f}")

    print("=== Step 8: Saving Artifacts ===")
    model_path = models_dir / MODEL_FILE
    with open(model_path, "wb") as f:
        pickle.dump(chosen_model, f)

    threshold_info = {
        "model_name": model_name,
        "best_threshold": float(best_thresh),
        "val_macro_f05": float(best_macro_f05),
        "blocking_recall_ceiling": float(pair_stats["blocking_recall_ceiling"]),
        "feature_names": FEATURE_COLS,
    }
    threshold_path = models_dir / THRESHOLD_FILE
    with open(threshold_path, "w", encoding="utf-8") as f:
        json.dump(threshold_info, f, indent=2)

    print(f"Saved model to: {model_path}")
    print(f"Saved threshold metadata to: {threshold_path}")

    return {
        "model": chosen_model,
        "model_path": model_path,
        "threshold_path": threshold_path,
        "best_threshold": best_thresh,
        "best_macro_f05": best_macro_f05,
        "pair_stats": pair_stats,
        "lgb_metrics": lgb_metrics,
        "lr_metrics": lr_metrics,
    }


def main():
    parser = argparse.ArgumentParser(description="Train Entity Matching model and sweep optimal threshold")
    parser.add_argument("--train-dir", default="dataset/train", help="Path to training TSVs directory")
    parser.add_argument("--models-dir", default="models", help="Directory to save matcher.pkl and threshold.json")
    parser.add_argument("--val-size", type=float, default=0.2, help="Validation split ratio")
    args = parser.parse_args()

    train_entity_matching_system(
        data_dir=args.train_dir,
        models_dir=args.models_dir,
        val_size=args.val_size,
    )


### 13. Inference Pipeline Function


In [ ]:
"""
Inference pipeline for test split and submission generation.
"""
import argparse
import json
import pickle
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd




def run_inference_pipeline(
    test_dir: Optional[Union[str, Path]] = None,
    test_dict: Optional[Dict[str, any]] = None,
    output_dir: Union[str, Path] = OUTPUT_DIR,
    models_dir: Union[str, Path] = MODELS_DIR,
    override_threshold: Optional[float] = None,
) -> Tuple[Path, Path]:
    """
    Executes complete inference:
    1. Loads test sources (S1, S2, S3)
    2. Runs BlockingEngine to generate candidates for all S1 test entities
    3. Exports output/candidate_pairs.tsv and candidate_pairs_internal.tsv
    4. Computes pairwise features
    5. Scores pairs with trained ML matcher
    6. Filters matches using optimal validation threshold
    7. Exports output/matching_results.tsv
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    models_dir = Path(models_dir)

    print("=== Step 1: Loading Test Data ===")
    if test_dict is None:
        if test_dir is None:
            raise ValueError("Must provide test_dir or test_dict")
        test_dict = load_split_datasets(test_dir, is_train=False)

    s1_df = test_dict["source1"]
    s2_df = test_dict["source2"]
    s3_df = test_dict["source3"]
    all_s1_ids = s1_df["entity_id"].tolist()
    print(f"Loaded test split: S1={len(s1_df)}, S2={len(s2_df)}, S3={len(s3_df)}")

    # Verify open-set countries
    countries_found = s1_df["country"].unique().tolist()
    print(f"Test S1 countries present (open-set verified): {countries_found}")

    print("=== Step 2: Running Blocking on Candidate Pool (S2 + S3) ===")
    blocking_engine = BlockingEngine()
    blocking_engine.fit(s2_df, s3_df)
    internal_cand_df, _ = blocking_engine.generate_candidates(s1_df)
    print(f"Generated {len(internal_cand_df)} total candidate pairs for {len(all_s1_ids)} S1 entities")

    print("=== Step 3: Exporting candidate_pairs.tsv ===")
    cand_path, int_cand_path = export_candidate_pairs(
        internal_cand_df,
        s1_entity_ids=all_s1_ids,
        output_dir=output_dir,
        save_internal_4col=True,
    )
    print(f"Exported candidate pairs to: {cand_path}")

    print("=== Step 4: Loading Model & Optimal Threshold ===")
    model_path = models_dir / MODEL_FILE
    threshold_path = models_dir / THRESHOLD_FILE

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    with open(threshold_path, "r", encoding="utf-8") as f:
        thresh_meta = json.load(f)

    threshold = (
        override_threshold if override_threshold is not None else float(thresh_meta["best_threshold"])
    )
    print(f"Loaded model ({thresh_meta.get('model_name', 'Model')}) with threshold: {threshold:.4f}")

    print("=== Step 5: Scoring Candidate Pairs ===")
    predictions_map: Dict[str, List[str]] = {s1: [] for s1 in all_s1_ids}

    if len(internal_cand_df) > 0:
        s1_dict = s1_df.set_index("entity_id").to_dict(orient="index")
        s23_combined = pd.concat([s2_df, s3_df], ignore_index=True)
        s23_dict = s23_combined.set_index("entity_id").to_dict(orient="index")

        X_cand_df = extract_features_for_pairs(internal_cand_df, s1_dict, s23_dict)
        X_test = X_cand_df[FEATURE_COLS].values

        probs = model.predict_proba(X_test)[:, 1]
        internal_cand_df["prob"] = probs

        # Filter accepted matches
        accepted_df = internal_cand_df[internal_cand_df["prob"] >= threshold]
        
        # Sort by probability descending within each S1
        accepted_sorted = accepted_df.sort_values(by=["source1_entity_id", "prob"], ascending=[True, False])
        
        grouped = accepted_sorted.groupby("source1_entity_id")["candidate_entity_id"].apply(list)
        for s1_id, matches in grouped.items():
            predictions_map[s1_id] = matches

    print("=== Step 6: Exporting matching_results.tsv ===")
    match_path = export_matching_results(
        predictions_map,
        s1_entity_ids=all_s1_ids,
        output_dir=output_dir,
    )
    print(f"Exported matching results to: {match_path}")

    non_empty_count = sum(1 for m in predictions_map.values() if len(m) > 0)
    print(f"Summary: {len(all_s1_ids)} S1 entities -> {non_empty_count} matched, "
          f"{len(all_s1_ids) - non_empty_count} singletons")

    return match_path, cand_path


def main():
    parser = argparse.ArgumentParser(description="Run inference on test set and output matching results")
    parser.add_argument("--test-dir", default="dataset/test", help="Path to test TSVs directory")
    parser.add_argument("--output-dir", default="output", help="Directory to save TSV outputs")
    parser.add_argument("--models-dir", default="models", help="Directory where matcher.pkl is saved")
    parser.add_argument("--threshold", type=float, default=None, help="Optional override for matching threshold")
    args = parser.parse_args()

    run_inference_pipeline(
        test_dir=args.test_dir,
        output_dir=args.output_dir,
        models_dir=args.models_dir,
        override_threshold=args.threshold,
    )


### 14. Submission Validator Functions


In [ ]:
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


### 15. Run Model Training & Threshold Optimization


In [ ]:
train_res = train_entity_matching_system(
    data_dir=TRAIN_DIR,
    models_dir=MODELS_DIR,
    val_size=0.25,
)
print(f"Optimal Threshold: {train_res['best_threshold']:.4f}")
print(f"Validation Macro F0.5: {train_res['best_macro_f05']:.4f}")


### 16. Run Inference on Test Split & Export TSVs


In [ ]:
match_path, cand_path = run_inference_pipeline(
    test_dir=TEST_DIR,
    output_dir=OUTPUT_DIR,
    models_dir=MODELS_DIR,
)
print(f"Candidate pairs: {cand_path}")
print(f"Matching results: {match_path}")


### 17. Validate Submission Output Files


In [ ]:
errors, warnings = validate(
    matching_path=str(match_path),
    candidate_path=str(cand_path),
    test_dir=str(TEST_DIR),
    check_ids=False,
)
if errors:
    print("Validation Errors:", errors)
else:
    print("PASS: Submission files are valid and ready to submit!")


### 18. Package Final Submission ZIP


In [ ]:
team_name = "antigravity_team"
zip_path = PROJECT_ROOT / f"{team_name}_submission.zip"
print(f"Packaging {zip_path}...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_DIR / MATCHING_RESULTS_FILE, arcname="output/matching_results.tsv")
    zf.write(OUTPUT_DIR / CANDIDATE_PAIRS_FILE, arcname="output/candidate_pairs.tsv")
    if (PROJECT_ROOT / "README.md").exists():
        zf.write(PROJECT_ROOT / "README.md", arcname="code/business_entity_resolution/README.md")
    if (PROJECT_ROOT / "requirements.txt").exists():
        zf.write(PROJECT_ROOT / "requirements.txt", arcname="code/business_entity_resolution/requirements.txt")
    if (PROJECT_ROOT / "Documentation_template.md").exists():
        zf.write(PROJECT_ROOT / "Documentation_template.md", arcname="Documentation_template.md")
print(f"Submission package successfully created: {zip_path.name}")
